# Creating Splits

In [ ]:
# 3️⃣ Cargar dataset desde archivo
data = np.load('hand_gesture_dataset.npz')
X = data['X']
y_labels = data['y']

# Convertir a one-hot
y = to_categorical(y_labels, num_classes=len(actions))

# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y_labels
)

print("X_train:", X_train.shape, "X_test:", X_test.shape)

X_train: (1045, 15, 63) X_test: (185, 15, 63)


# Creating Model

In [ ]:
MODEL_EXPORT_NAME = 'hand_gesture_model.h5'

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GRU, TimeDistributed

model = Sequential([
    TimeDistributed(Dense(64, activation='relu'),
                    input_shape=(sequence_length, X.shape[2])),
    Dropout(0.3),

    GRU(64),
    Dense(32, activation='relu'),
    Dense(len(actions), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

/home/luis/Documents/projects/python/HandActionDetectionModel/venv/lib/python3.12/site-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed_1              │ (None, 15, 64)         │         4,096 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 15, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,235 (122.01 KB)

 Trainable params: 31,235 (122.01 KB)

 Non-trainable params: 0 (0.00 B)

# Training

In [ ]:
model.fit(
    X_train, y_train,
    epochs=100,
    validation_split=0.2,
    batch_size=16
)

model.save(MODEL_EXPORT_NAME)

Epoch 1/100
53/53 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 0.5766 - loss: 0.8783 - val_accuracy: 0.6364 - val_loss: 0.6559
Epoch 2/100
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8002 - loss: 0.4957 - val_accuracy: 0.9856 - val_loss: 0.2050
Epoch 3/100
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9689 - loss: 0.1204 - val_accuracy: 1.0000 - val_loss: 0.0133
Epoch 4/100
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9988 - loss: 0.0145 - val_accuracy: 1.0000 - val_loss: 0.0024
Epoch 5/100
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 1.0000 - loss: 0.0031 - val_accuracy: 1.0000 - val_loss: 9.7936e-04
Epoch 6/100
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9952 - loss: 0.0212 - val_accuracy: 1.0000 - val_loss: 0.0030
Epoch 7/100
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9761 - loss: 0.1030 - val_accuracy: 1.0000 - val_loss: 0.0049
Epoch 8/100
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 1.0000 - loss: 0.0035 - val_accuracy:

# Metrics

In [ ]:
# -------------------------
# 1️⃣ Importaciones y configuraciones
# -------------------------
import numpy as np
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, GRU, TimeDistributed
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

# Dataset y parámetros
DATASET_PATH = 'hand_gesture_dataset.npz'
# ACTIONS = ["grab", "release", "turpinch", "unpinch", "screw","none"]
ACTIONS = ["A", "B", "C", "D", "E", "F"]

SEQUENCE_LENGTH = 10
MODEL_EXPORT_NAME = 'hand_gesture_model.h5'

# -------------------------
# 2️⃣ Cargar dataset desde archivo
# -------------------------
data = np.load(DATASET_PATH)
X = data['X']  # forma original: (num_samples, sequence_length, 21, 3)
y_labels = data['y']

# Aplanar landmarks de cada frame: (21,3) -> (63)
num_samples = X.shape[0]
X = X.reshape(num_samples, SEQUENCE_LENGTH, -1)  # ahora (num_samples, sequence_length, 63)

# Convertir labels a one-hot
y = to_categorical(y_labels, num_classes=len(ACTIONS))

# -------------------------
# 3️⃣ Train / Test split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y_labels, random_state=42
)

print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("y_train:", y_train.shape, "y_test:", y_test.shape)

# -------------------------
# 4️⃣ Crear modelo
# -------------------------
model = Sequential([
    TimeDistributed(Dense(64, activation='relu'), input_shape=(SEQUENCE_LENGTH, X.shape[2])),
    Dropout(0.3),

    GRU(64, return_sequences=True),
    GRU(128, return_sequences=True),
    GRU(64, return_sequences=False),
    Dense(32, activation='relu'),
    Dense(len(ACTIONS), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# -------------------------
# 5️⃣ Entrenamiento
# -------------------------
history = model.fit(
    X_train, y_train,
    epochs=100,
    validation_split=0.2,
    batch_size=16,
    verbose=1
)

# Guardar modelo entrenado
model.save(MODEL_EXPORT_NAME)
print(f"Modelo guardado en {MODEL_EXPORT_NAME}")

# -------------------------
# 6️⃣ Evaluación / Métricas
# -------------------------
model = load_model(MODEL_EXPORT_NAME)

y_pred = np.argmax(model.predict(X_test), axis=1)
y_true = np.argmax(y_test, axis=1)

acc = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

print(f"Accuracy en test set: {acc:.4f}")
print("Matriz de confusión:")
print(cm)


X_train: (550, 10, 126) X_test: (98, 10, 126)
y_train: (550, 6) y_test: (98, 6)


/home/luis/Documents/projects/python/HandActionDetectionModel/venv/lib/python3.12/site-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed_1              │ (None, 10, 64)         │         8,128 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 10, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ (None, 10, 64)         │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_4 (GRU)                     │ (None, 10, 128)        │        74,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_5 (GRU)                     │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 147,110 (574.65 KB)

 Trainable params: 147,110 (574.65 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 6s 41ms/step - accuracy: 0.4114 - loss: 1.4745 - val_accuracy: 0.4636 - val_loss: 1.0319
Epoch 2/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6045 - loss: 0.9818 - val_accuracy: 0.7455 - val_loss: 0.7036
Epoch 3/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7318 - loss: 0.6425 - val_accuracy: 0.8909 - val_loss: 0.4474
Epoch 4/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8432 - loss: 0.4451 - val_accuracy: 0.8273 - val_loss: 0.3830
Epoch 5/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8705 - loss: 0.3536 - val_accuracy: 0.8455 - val_loss: 0.3133
Epoch 6/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.8977 - loss: 0.2837 - val_accuracy: 0.9182 - val_loss: 0.2779
Epoch 7/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9182 - loss: 0.2299 - val_accuracy: 0.9455 - val_loss: 0.1717
Epoch 8/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9455 - loss: 0.1715 - val_accuracy: 0.